# Melting DataFrames Practice Exercises — English Premier League

Pivot tables are great to *read*, but wide tables are awkward to *analyse further*. In these exercises you'll take wide, pivoted tables and **unpivot** them back into tidy long form with `melt()`. Work through them **in order**.

Each question describes the *outcome* you need — figure out which arguments fit. The `*(covers: ...)*` line points you back to your `9.meltingDataFrames` lesson notebook.

Run the setup cell below first, then write your answers in the empty cell under each exercise. When you're done, come back and I'll check them.

In [1]:
import numpy as np
import pandas as pd

premier_league = pd.read_excel("../retailFiles/premier_league_games_full.xlsx")

# A wide pivot table we'll reshape throughout: total home goals,
# rows = home team, columns = season (three teams)
goals_pivot = premier_league.query(
    "HomeTeam in ['Liverpool', 'Manchester United', 'Tottenham Hotspur']"
).pivot_table(index="HomeTeam",
              columns="season",
              values="HomeGoals",
              aggfunc="sum"
              )
goals_pivot

season,2008/2009,2009/2010,2010/2011,2011/2012,2012/2013,2013/2014,2014/2015,2015/2016
HomeTeam,,,,,,,,
Liverpool,41,43,37,24,33,53,30,33
Manchester United,43,52,49,52,45,29,41,27
Tottenham Hotspur,21,40,30,39,29,30,31,35


## Exercise 1 — Unpivot Everything
*(covers: meltingDataFrames — reset_index + melt)*

1. `goals_pivot` keeps the home team in its **row index**. First move it back into a regular column so `melt()` can see it.
2. Now `melt()` the result with **no arguments** and look at what you get — notice how *every* column, including the team name, gets stacked into one long column.

In [14]:
# 1. Move HomeTeam out of the index into a column
goals_pivot.reset_index()

# 2. melt() with no arguments
goals_pivot.reset_index().melt()

,season,value
0,HomeTeam,Liverpool
1,HomeTeam,Manchester United
2,HomeTeam,Tottenham Hotspur
3,2008/2009,41
4,2008/2009,43
5,2008/2009,21
6,2009/2010,43
7,2009/2010,52
8,2009/2010,40
9,2010/2011,37


## Exercise 2 — Keep an Identifier Column
*(covers: meltingDataFrames — id_vars)*

Melt `goals_pivot` again (after resetting its index), but this time **keep `HomeTeam` as an identifier** so each row ends up as one team, one season, and that team's home goals for the season.

In [15]:
# Melt while keeping HomeTeam as the identifier column
goals_pivot.reset_index().melt(id_vars="HomeTeam")

,HomeTeam,season,value
0,Liverpool,2008/2009,41
1,Manchester United,2008/2009,43
2,Tottenham Hotspur,2008/2009,21
3,Liverpool,2009/2010,43
4,Manchester United,2009/2010,52
5,Tottenham Hotspur,2009/2010,40
6,Liverpool,2010/2011,37
7,Manchester United,2010/2011,49
8,Tottenham Hotspur,2010/2011,30
9,Liverpool,2011/2012,24


## Exercise 3 — Name Your Output Columns
*(covers: meltingDataFrames — var_name / value_name)*

Melt `goals_pivot` keeping `HomeTeam` as the identifier, but give the two new columns meaningful names: the column holding the season labels should be called `"season"`, and the column holding the numbers should be called `"home_goals"`. Save the result as `goals_long`.

In [20]:
# Melt with named var/value columns -> goals_long
goals_long = goals_pivot.reset_index().melt(id_vars="HomeTeam",
                               value_name="home_goals")


## Exercise 4 — Unpivot Only Some Columns
*(covers: meltingDataFrames — value_vars)*

Sometimes you only want a few of the seasons. Melt `goals_pivot` (index reset, `HomeTeam` as the identifier) but **only unpivot the first three seasons** — `"2008/2009"`, `"2009/2010"`, and `"2010/2011"` — naming the value column `"home_goals"`.

In [42]:
# Melt only the three named seasons
goals_pivot.reset_index().melt(id_vars="HomeTeam",
                               value_vars=["2008/2009", "2009/2010", "2010/2011"],
                               value_name="home_goals")

,HomeTeam,season,home_goals
0,Liverpool,2008/2009,41
1,Manchester United,2008/2009,43
2,Tottenham Hotspur,2008/2009,21
3,Liverpool,2009/2010,43
4,Manchester United,2009/2010,52
5,Tottenham Hotspur,2009/2010,40
6,Liverpool,2010/2011,37
7,Manchester United,2010/2011,49
8,Tottenham Hotspur,2010/2011,30


## Exercise 5 — Putting It Together: Pivot, Then Tidy Back
*(covers: a mix of PivotTables + meltingDataFrames)*

1. Build a fresh pivot table of **total home goals** for **all** teams (rows = home team, columns = season) and save it as `all_teams`.
2. Reset its index and `melt()` it into tidy long form with columns `HomeTeam`, `"season"`, and `"home_goals"`.
3. Some team/season combinations never happened (a team wasn't in the league that year), so they came through as missing — **drop those rows**.
4. Finally, sort the tidy table by `home_goals` from **highest to lowest** and show the **top 10** team-seasons.

In [41]:
# 1. Pivot: total home goals for all teams -> all_teams
all_teams = premier_league.pivot_table(index="HomeTeam",
                           columns="season",
                           values="HomeGoals",
                           aggfunc="sum")

# 2. Reset index and melt into tidy long form
all_teams.reset_index().melt(id_vars="HomeTeam",
                             value_name="home_goals")

# 3. Drop team/season combinations that never happened (missing values)
all_teams.reset_index().melt(id_vars="HomeTeam",
                             value_name="home_goals").dropna()

# 4. Sort by home_goals descending and show the top 10
all_teams.reset_index().melt(id_vars="HomeTeam",
                             value_name="home_goals").dropna().sort_values(by="home_goals", ascending=False).iloc[:10]

,HomeTeam,season,home_goals
43,Chelsea,2009/2010,68.0
186,Manchester City,2013/2014,63.0
118,Manchester City,2011/2012,55.0
185,Liverpool,2013/2014,53.0
51,Manchester United,2009/2010,52.0
119,Manchester United,2011/2012,52.0
85,Manchester United,2010/2011,49.0
34,Arsenal,2009/2010,48.0
136,Arsenal,2012/2013,47.0
254,Manchester City,2015/2016,47.0
